In [ ]:
!pip install -q transformers datasets peft scikit-learn torch accelerate spacy scipy xlsxwriter seaborn bitsandbytes
!python -m spacy download en_core_web_sm -q

import os, sys, math, gc, json, warnings
from itertools import product
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
from scipy.special import logsumexp
import spacy
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling, set_seed
from peft import LoraConfig, get_peft_model
from datasets import Dataset

warnings.filterwarnings('ignore')
os.environ["WANDB_DISABLED"] = "true"

plt.rcParams.update({"font.family": "serif", "font.serif": ["cmr10", "DejaVu Serif", "Bitstream Vera Serif"], "mathtext.fontset": "cm", "axes.labelsize": 14, "font.size": 12, "legend.fontsize": 12, "xtick.labelsize": 12, "ytick.labelsize": 12, "figure.figsize": [10, 8], "image.cmap": "viridis", "axes.formatter.use_mathtext": True})

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

OUTS = 'M_kw_nM_Advanced'
BASE_DRIVE_PATH = f'/content/drive/My Drive/University_MATH/{OUTS}' if IN_COLAB else f'./University_MATH/{OUTS}'
DATASET_FULL_PATH = f'/content/drive/My Drive/University_MATH/x.json' if IN_COLAB else 'dummy.json'
device = "cuda" if torch.cuda.is_available() else "cpu"

QUICK_CONFIG = {"run_config": {"experiment_name": "Exp_M_QQQ_Advanced", "num_replicas": 2, "device": device, "metric_for_hypothesis_testing": "RLCT"}, "data_config": {"dataset_path": DATASET_FULL_PATH, "samples_in_balance": 15, "finetune_budget": 50, "max_seq_len": 64}, "model_config": {"base_model_id": "openai-community/gpt2", "lora_config": {"r": 8, "lora_alpha": 16, "target_modules": ["c_attn"]}, "ir_qlora_config": {"tau_lambda": 0.1, "tau_n": 10}}, "training_config": {"max_steps": 10, "learning_rate": 5e-5, "per_device_train_batch_size": 4, "logging_steps": 1}, "rlct_config": {"max_rlct_estimation_steps": 20, "sgld_lr": 1e-4}, "moi_grid": {"betas": [0.1, 1.0, 5.0], "temperatures": [0.7, 1.0], "repetition_penalties": [1.0, 1.2], "top_ks": [0, 50]}}
ACTIVE_CONFIG = QUICK_CONFIG

class EXPERIMENTACION_ANOVA:
    def __init__(self, master_config: dict): self.config = master_config; self.device = torch.device(self.config['run_config']['device']); self.base_dir = BASE_DRIVE_PATH; os.makedirs(self.base_dir, exist_ok=True); set_seed(42); print(f"=== INICIALIZANDO EXPERIMENTO AVANZADO: {self.config['run_config']['experiment_name']} ==="); print(f"Directorio de salida: {self.base_dir}"); __import__("subprocess").call([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"]); self.nlp = spacy.load("en_core_web_sm"); self.tokenizer = AutoTokenizer.from_pretrained(self.config['model_config']['base_model_id']); self.tokenizer.pad_token = self.tokenizer.eos_token if self.tokenizer.pad_token is None else self.tokenizer.pad_token; self.doc_gen = self.Subclase_Documentacion(self); self.data_handler = self.Subclase_FactorLinguistico(self); self.rlct_estimator = self.Subclase_EstimadorRLCT(self); self.moi_evaluator = self.Subclase_MOI_and_GRID(self); self.visualizer = self.Subclase_VisualizadorAvanzado(self); self.stats = self.Subclase_AnalisisEstadistico(self); self.doc_gen.generar_fundamentos_matematicos()

    class Subclase_Documentacion:
        def __init__(self, parent): self.parent = parent; self.txt_dir = os.path.join(self.parent.base_dir, "Fundamentos"); os.makedirs(self.txt_dir, exist_ok=True)
        def generar_fundamentos_matematicos(self): open(os.path.join(self.txt_dir, "Math_Foundations.txt"), "w", encoding="utf-8").write(r"\section{Fundamentos Matemáticos del Experimento}\n\n\subsection{Factor Lingüístico (ASL)}\nLa complejidad sintáctica se particiona utilizando la Longitud Media de la Frase (ASL):\n\begin{equation}\n\text{ASL} = \frac{\text{Total Tokens}}{\text{Total Frases}}\n\end{equation}\nDefiniendo los tratamientos: Simple\_Syntax, Middle\_Syntax y Complex\_Syntax.\n\n\subsection{Método IR-QLoRA (Information Retention QLoRA)}\nUtiliza Information Calibration Quantization (ICQ) maximizando la entropía $H$:\n\begin{equation}\nH = - \sum P(q_i) \log_2 P(q_i)\n\end{equation}\nY una Information Elastic Connection (IEC) para el flujo residual:\n\begin{equation}\nU_{IEC}(x) = x' \cdot \ell_2 + \frac{\beta_2}{o} \cdot x_{res2}\n\end{equation}\n\n\subsection{Estimador RLCT y Geometría Local}\nDefinimos temperaturas inversas $\beta_1 = 1/\log n$ y $\beta_2 = c/\log n$. La energía esperada re-ponderada con el estimador Watanabe:\n\begin{equation}\n\mathbb{E}^{\beta_2}_w[nL_n(w)] = \frac{\mathbb{E}^{\beta_1}_w \left[ nL_n(w) \exp\left( -(\beta_2 - \beta_1) nL_n(w) \right) \right]}{\mathbb{E}^{\beta_1}_w \left[ \exp\left( -(\beta_2 - \beta_1) nL_n(w) \right) \right]}\n\end{equation}\nLa fluctuación de la energía $\Delta \mathcal{L} \approx |E_t - E_{t-1}|$ aproxima la divergencia KL en la topología singular del espacio de pesos durante las dinámicas estocásticas de Langevin (SGLD).\n\n\subsection{Mezcla de Entradas Bayesiana (Mixture of Inputs - MoI) Avanzada}\nModelo Dirichlet-Multinomial para amalgamar embeddings. La distribución a priori del modelo $p_t$ es alterada por la temperatura $T$ y la penalización de repetición $\rho$. La media a posteriori del peso de mezcla $w_{t,i}$ dado un parámetro de concentración $\beta$ es:\n\begin{equation}\nw_{t,i} = \frac{H(p_t) \cdot p_{t,i} + (\beta + 1 - H(p_t)) \cdot y_{t,i}}{\beta + 1}\n\end{equation}\nEl embedding continuo se genera como $h_t = \sum_{i=1}^V w_{t,i} e_i\n".strip())

    class Subclase_FactorLinguistico:
        def __init__(self, parent): self.parent = parent
        def preparar_dataset(self): cfg = self.parent.config['data_config']; df = pd.DataFrame(json.load(open(cfg['dataset_path']))) if os.path.exists(cfg['dataset_path']) else pd.DataFrame({"instruction": ["Explain this simple. " * np.random.randint(1, 4) for _ in range(cfg['samples_in_balance'] * 5)], "response": ["Response. " * 5 for _ in range(cfg['samples_in_balance'] * 5)]}); df_asl = pd.DataFrame([{**row, 'asl': len([t for t in doc if t.is_alpha]) / len(list(doc.sents)) if len(list(doc.sents)) > 0 else 0.0} for idx, row in df.iterrows() for doc in [self.parent.nlp(str(row['instruction']))]]).dropna(subset=['asl']).sort_values('asl').reset_index(drop=True); n_samples = max(1, len(df_asl) // 3) if len(df_asl) < cfg['samples_in_balance'] * 3 else cfg['samples_in_balance']; return {"Simple_Syntax": Dataset.from_pandas(df_asl.head(n_samples)), "Middle_Syntax": Dataset.from_pandas(df_asl.iloc[(len(df_asl) - n_samples) // 2 : (len(df_asl) - n_samples) // 2 + n_samples]), "Complex_Syntax": Dataset.from_pandas(df_asl.tail(n_samples))}

    class Subclase_MetodosAdaptacion:
        class IRQLoraLinear(nn.Module):
            def __init__(self, base_layer, r, alpha, tau_lambda, tau_n): super().__init__(); self.in_features, self.out_features, w = (base_layer.in_features, base_layer.out_features, base_layer.weight.data.clone().float()) if hasattr(base_layer, "in_features") else (base_layer.weight.shape[0], base_layer.weight.shape[1], base_layer.weight.data.clone().t().float()); best_tau = [max([(-torch.sum((p := (hist := torch.histc(w - (tau0 + f*(w - tau0).abs().max()), bins=256)) / (hist.sum() + 1e-9))[p > 0] * torch.log2(p[p > 0])), tau0 + f*(w - tau0).abs().max()) for f in np.linspace(-tau_lambda*0.5, tau_lambda*0.5, 2*tau_n+1)], key=lambda x: x[0])[1] for tau0 in [w.median()]][0]; self.tau = nn.Parameter(torch.tensor(best_tau, dtype=base_layer.weight.dtype), requires_grad=False); self.weight = nn.Parameter((w - self.tau).to(base_layer.weight.dtype), requires_grad=False); self.bias = nn.Parameter(base_layer.bias.data.clone(), requires_grad=False) if base_layer.bias is not None else self.register_parameter('bias', None); self.lora_A = nn.Linear(self.in_features, r, bias=False); self.lora_B = nn.Linear(r, self.out_features, bias=False); nn.init.normal_(self.lora_A.weight, std=1/r); nn.init.zeros_(self.lora_B.weight); self.scaling = alpha / r; self.beta1 = nn.Parameter(torch.zeros(1), requires_grad=True); self.beta2 = nn.Parameter(torch.zeros(1), requires_grad=True)
            def forward(self, x): return F.linear(x, self.weight + self.tau, self.bias) + (self.lora_B(self.lora_A(x) + self.beta1 * x.mean(dim=-1, keepdim=True)) + self.beta2 * (self.lora_A(x) + self.beta1 * x.mean(dim=-1, keepdim=True)).mean(dim=-1, keepdim=True)) * self.scaling

        @classmethod
        def apply_ir_qlora(cls, model, config): lora_cfg, ir_cfg = config['model_config']['lora_config'], config['model_config'].get('ir_qlora_config', {"tau_lambda": 0.1, "tau_n": 10}); [setattr(model.get_submodule('.'.join(name.split('.')[:-1])) if '.' in name else model, name.split('.')[-1], cls.IRQLoraLinear(module, lora_cfg['r'], lora_cfg['lora_alpha'], ir_cfg['tau_lambda'], ir_cfg['tau_n'])) for name, module in list(model.named_modules()) if type(module).__name__ in ['Linear', 'Conv1D'] and any(t in name for t in lora_cfg['target_modules'])]; getattr(model, "enable_input_require_grads", lambda: None)(); [param.requires_grad_(True) if 'lora' in name or 'beta' in name else param.requires_grad_(False) for name, param in model.named_parameters()]; return model

    class Subclase_EstimadorRLCT:
        def __init__(self, parent): self.parent = parent
        class SGLD(torch.optim.Optimizer):
            def __init__(self, params, lr=1e-4, temperature=1.): super().__init__(params, dict(lr=lr, temperature=temperature, noise_level=1.0))
            def step(self): prev_grad = torch.is_grad_enabled(); torch.set_grad_enabled(False); [p.add_(p.grad, alpha=-0.5 * group['lr'] / group['temperature']).add_(torch.normal(0., group['noise_level'], size=p.size(), device=p.device), alpha=math.sqrt(group['lr'])) for group in self.param_groups for p in group['params'] if p.grad is not None]; torch.set_grad_enabled(prev_grad)

        def estimate_rlct(self, model, dataset): cfg, n = self.parent.config['rlct_config'], max(1, len(dataset)); beta1, beta2, optim, energies, lambdas = 1.0 / np.log(n), 1.5 / np.log(n), self.SGLD(model.parameters(), lr=cfg['sgld_lr'], temperature=1.0 / (1.0 / np.log(n))), [], []; model.train(); [(optim.zero_grad(), loss := model(**(inputs := self.parent.tokenizer(f"{dataset[i % n]['instruction']}\n{dataset[i % n].get('response', '')}", return_tensors="pt", truncation=True, max_length=64).to(self.parent.device)), labels=inputs.input_ids).loss, loss.backward(), optim.step(), energies.append(loss.item() * n), lambdas.append(max(0.0, lam if not np.isnan(lam := (np.mean(V) - np.sum(np.exp((lw := -(beta2 - beta1) * V) - logsumexp(lw)) * V)) / (1/beta1 - 1/beta2)) else 0.0)) if len(energies) > 2 and (V := np.array(energies)) is not None else None) for i in range(min(n, cfg['max_rlct_estimation_steps']))]; return (lambdas[-1] if lambdas else 0.0), lambdas, energies

    class Subclase_MOI_and_GRID:
        def __init__(self, parent): self.parent = parent
        class BayesianMoI_DirichletMultinomial:
            def __init__(self, model, tokenizer, beta, temp, rep_penalty, top_k): self.model, self.tokenizer, self.beta, self.T, self.rho, self.top_k, self.vocab_size, self.embed_layer = model, tokenizer, beta, temp, rep_penalty, top_k, model.config.vocab_size, model.get_input_embeddings() if hasattr(model, 'get_input_embeddings') and model.get_input_embeddings() is not None else (model.base_model.get_input_embeddings() if hasattr(model, 'base_model') and model.base_model.get_input_embeddings() is not None else list(model.modules())[1])
            def compute_posterior_embedding(self, logits, sampled_token_id): logits_adj = logits / (self.T + 1e-7); _ = logits_adj.masked_fill_(logits_adj < torch.topk(logits_adj, min(self.top_k, logits_adj.size(-1)))[0][..., -1, None], -float('Inf')) if self.top_k > 0 else None; probs = torch.softmax(logits_adj, dim=-1); return torch.matmul(((-torch.sum(probs * torch.log(probs + 1e-10), dim=-1, keepdim=True) / math.log(self.vocab_size)) * probs + (self.beta + 1.0 - (-torch.sum(probs * torch.log(probs + 1e-10), dim=-1, keepdim=True) / math.log(self.vocab_size))) * torch.zeros_like(probs).scatter_(1, sampled_token_id.unsqueeze(-1), 1.0)) / (self.beta + 1.0), self.embed_layer.weight).unsqueeze(1)
            def generar_texto_bayesiano(self, input_text, max_new_tokens=20): self.model.eval(); inputs = self.tokenizer(input_text, return_tensors="pt").to(self.model.device); current_embeds, generated_ids = self.embed_layer(inputs.input_ids), inputs.input_ids.clone(); prev_grad = torch.is_grad_enabled(); torch.set_grad_enabled(False); [(next_token_logits := self.model(inputs_embeds=current_embeds).logits[:, -1, :].clone(), [next_token_logits[0].__setitem__(token, next_token_logits[0, token] / self.rho if next_token_logits[0, token] > 0 else next_token_logits[0, token] * self.rho) for token in set(generated_ids[0].tolist())] if self.rho > 1.0 else None, next_token_id := torch.argmax(next_token_logits, dim=-1), current_embeds := torch.cat([current_embeds, self.compute_posterior_embedding(next_token_logits, next_token_id)], dim=1), generated_ids := torch.cat([generated_ids, next_token_id.unsqueeze(0)], dim=-1)) for _ in range(max_new_tokens) if generated_ids[0, -1].item() != self.tokenizer.eos_token_id]; torch.set_grad_enabled(prev_grad); return generated_ids

        def evaluar_grid(self, model, dataset): cfg, prompt = self.parent.config['moi_grid'], dataset[0]['instruction'] if len(dataset) > 0 else "Explain gravity."; prev_grad = torch.is_grad_enabled(); torch.set_grad_enabled(False); resultados = [{"beta": b, "temperature": t, "repetition_penalty": r, "top_k": k, "ppl": (lambda dec: torch.exp(model(self.parent.tokenizer(dec, return_tensors="pt").input_ids.to(self.parent.device), labels=self.parent.tokenizer(dec, return_tensors="pt").input_ids.to(self.parent.device)).loss).item() if not torch.isnan(model(self.parent.tokenizer(dec, return_tensors="pt").input_ids.to(self.parent.device), labels=self.parent.tokenizer(dec, return_tensors="pt").input_ids.to(self.parent.device)).loss) else float('inf'))(self.parent.tokenizer.decode(self.BayesianMoI_DirichletMultinomial(model, self.parent.tokenizer, b, t, r, k).generar_texto_bayesiano(prompt, max_new_tokens=15)[0], skip_special_tokens=True))} for b, t, r, k in product(cfg['betas'], cfg['temperatures'], cfg['repetition_penalties'], cfg['top_ks'])]; torch.set_grad_enabled(prev_grad); df = pd.DataFrame(resultados); return df['ppl'].min(), df

    class Subclase_VisualizadorAvanzado:
        def __init__(self, parent): self.parent = parent
        def plot_loss_history(self, log_history, run_id, output_dir): losses, steps = [x['loss'] for x in log_history if 'loss' in x], [x['step'] for x in log_history if 'loss' in x]; (plt.figure(figsize=(10, 6)), plt.plot(steps, losses, marker='o', linestyle='-', color='darkblue', linewidth=1.5, label='Training Loss'), plt.yscale('log'), plt.title(f"Run {run_id}: Optimization Convergence $\\mathcal{{L}}(\\theta)$"), plt.xlabel("Training Steps $t$"), plt.ylabel(r"Loss $\log \mathcal{L}$"), plt.legend(), plt.tight_layout(), plt.savefig(os.path.join(output_dir, "training_loss_evolution.png")), plt.close()) if losses else None
        def plot_rlct_evolution(self, lambda_history, run_id, output_dir): plt.figure(figsize=(10, 6)); plt.plot(lambda_history, color='purple', linewidth=1.5); plt.title(rf"Run {run_id}: RLCT Convergence $\lambda_t$ (WBIC)"); plt.xlabel("SGLD Steps (Cumulative)"); plt.ylabel(r"Estimated RLCT $\hat{\lambda}$"); plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.savefig(os.path.join(output_dir, "rlct_evolution_lambda.png")); plt.close()
        def plot_kl_divergence(self, energy_history, run_id, output_dir): plt.figure(figsize=(10, 6)); plt.plot(np.abs(np.diff(energy_history)), color='teal', alpha=0.7, linewidth=1); plt.title(rf"Run {run_id}: Local Divergence $\Delta \mathcal{{L}} \approx |E_t - E_{{t-1}}|$"); plt.xlabel("SGLD Steps"); plt.ylabel(r"Energy Fluctuation"); plt.yscale('log'); plt.tight_layout(); plt.savefig(os.path.join(output_dir, "rlct_kl_divergence.png")); plt.close()
        def plot_moi_analysis(self, df, run_id, output_dir): (fig := plt.figure(figsize=(14, 10)), fig.suptitle(f"Run {run_id}: MoI Sensitivity Analysis (Factorial)", fontsize=14), [sns.boxplot(x=col, y='ppl', data=df, ax=fig.add_subplot(2, 2, i+1), palette="viridis").set(xlabel=label, ylabel=r"Perplexity $PPL_{MoI}$ (Log)", yscale='log') for i, (col, label) in enumerate([('beta', r'Concentration $\beta$'), ('temperature', r'Temperature $T$'), ('repetition_penalty', r'Repetition Penalty $\rho$'), ('top_k', r'Top-K Filtering')]) if col in df.columns], plt.tight_layout(rect=[0, 0.03, 1, 0.95]), plt.savefig(os.path.join(output_dir, "moi_boxplots_factorial.png")), plt.close(), (plt.figure(figsize=(8, 6)), sns.heatmap(df.pivot_table(index='temperature', columns='beta', values='ppl', aggfunc='mean'), annot=True, fmt=".1f", cmap="viridis_r", cbar_kws={'label': 'Mean PPL'}), plt.title(f"Run {run_id}: MoI Stability Landscape ($T$ vs $\\beta$)"), plt.xlabel(r'Concentration $\beta$'), plt.ylabel(r'Temperature $T$'), plt.tight_layout(), plt.savefig(os.path.join(output_dir, "moi_heatmap_interaction.png")), plt.close()) if 'beta' in df.columns and 'temperature' in df.columns else None) if not df.empty else None

    class Subclase_AnalisisEstadistico:
        def __init__(self, parent): self.parent = parent
        def run_kruskal_wallis(self, df_results): metric, txt_path = self.parent.config['run_config']['metric_for_hypothesis_testing'], os.path.join(self.parent.base_dir, "Sustentacion_Estadistica_Matematica.txt"); g_asl, g_met = [g[metric].values for _, g in df_results.groupby('ASL_Treatment') if len(g) > 0], [g[metric].values for _, g in df_results.groupby('Method') if len(g) > 0]; stat_a, p_a = kruskal(*g_asl) if len(g_asl) > 1 else (0, 1); stat_b, p_b = kruskal(*g_met) if len(g_met) > 1 else (0, 1); open(txt_path, "w", encoding="utf-8").write((r"\section{Sustentación Matemática de Conclusiones}\n\begin{block}{Modelo ANOVA Adaptativo}\n\begin{itemize}\n\item \textbf{Factor $A:\alpha$ (Tratamiento Lingüístico ASL)}: 3 niveles (Simple, Middle, Complex).\n\item \textbf{Factor $B:\beta$ (Método de Fine-Tuning)}: 2 niveles (LORA Standard, IR-QLoRA).\n\end{itemize}\nEcuación del Modelo Lineal Generalizado robusto:\n\begin{equation}\n\text{%s} = \mu + \alpha_i + \beta_j + (\alpha\beta)_{ij} + \epsilon_{ijk}\n\end{equation}\n\end{block}\n\n\textbf{Resultados de la Prueba No Paramétrica Kruskal-Wallis para %s:}\n\begin{itemize}\n\item Factor A (ASL): $H = %.4f$, valor-p $= %.4f$\n\item Factor B (Método): $H = %.4f$, valor-p $= %.4f$\n\end{itemize}\n\n\textbf{Conclusiones Matemáticas y Topológicas:}\n" % (metric, metric, stat_a, p_a, stat_b, p_b) + ("\nExiste evidencia de que la longitud media de la frase (ASL) altera la topología singular del modelo (RLCT), reflejándose en las fluctuaciones locales de energía $\Delta \mathcal{L}$.\n" if p_a < 0.05 else "") + ("\nEl método IR-QLoRA, mediante su cuantización ICQ orientada a entropía y su inyección IEC, genera un paisaje de pérdida significativamente diferente al LoRA estándar, favoreciendo distintas estabilidades en la Mixture of Inputs (MoI).\n" if p_b < 0.05 else "")).strip())

    def run(self): cfg, pools, global_results = self.config, self.data_handler.preparar_dataset(), []; [(run_dir := os.path.join(self.base_dir, (run_id := f"Rep{rep}_{treat_name}_{method}")), os.makedirs(run_dir, exist_ok=True), print(f"\n[{run_id}] Iniciando Pipeline..."), base_model := AutoModelForCausalLM.from_pretrained(cfg['model_config']['base_model_id']).to(self.device), model := (get_peft_model(base_model, LoraConfig(task_type="CAUSAL_LM", **cfg['model_config']['lora_config'])).apply(lambda m: getattr(m, "enable_input_require_grads", lambda: None)()) if method == "LORA_Standard" else self.Subclase_MetodosAdaptacion.apply_ir_qlora(base_model, cfg)), trainer := Trainer(model=model, args=TrainingArguments(output_dir=os.path.join(run_dir, "ckpt"), max_steps=cfg['training_config']['max_steps'], learning_rate=cfg['training_config']['learning_rate'], per_device_train_batch_size=cfg['training_config']['per_device_train_batch_size'], report_to="none", save_strategy="no", remove_unused_columns=True), train_dataset=pool_ds.map(lambda batch: self.tokenizer([f"{i}\n{r}" for i, r in zip(batch['instruction'], batch.get('response', [''] * len(batch['instruction'])))], truncation=True, padding="max_length", max_length=cfg['data_config']['max_seq_len']), batched=True), data_collator=DataCollatorForLanguageModeling(self.tokenizer, mlm=False)), trainer.train(), self.visualizer.plot_loss_history(trainer.state.log_history, run_id, run_dir), model.eval(), prev_grad := torch.is_grad_enabled(), torch.set_grad_enabled(False), losses := [model(**self.tokenizer(pool_ds[i]['instruction'], return_tensors='pt', truncation=True, max_length=64).to(self.device), labels=self.tokenizer(pool_ds[i]['instruction'], return_tensors='pt', truncation=True, max_length=64).to(self.device).input_ids).loss.item() for i in range(min(5, len(pool_ds)))], valid_losses := [l for l in losses if not math.isnan(l)], ppl := math.exp(min(sum(valid_losses)/len(valid_losses) if valid_losses else 100.0, 20)), torch.set_grad_enabled(prev_grad), rlct_data := self.rlct_estimator.estimate_rlct(model, pool_ds), self.visualizer.plot_rlct_evolution(rlct_data[1], run_id, run_dir), self.visualizer.plot_kl_divergence(rlct_data[2], run_id, run_dir), moi_data := self.moi_evaluator.evaluar_grid(model, pool_ds), self.visualizer.plot_moi_analysis(moi_data[1], run_id, run_dir), global_results.append({"Replica": rep, "ASL_Treatment": treat_name, "Method": method, "RLCT": rlct_data[0], "PPL": ppl, "MoI_PPL": moi_data[0]}), gc.collect(), torch.cuda.empty_cache()) for rep in range(cfg['run_config']['num_replicas']) for treat_name, pool_ds in pools.items() for method in ["LORA_Standard", "IR_QLoRA"]]; pd.DataFrame(global_results).to_excel(os.path.join(self.base_dir, "RESULTS.xlsx"), index=False); self.stats.run_kruskal_wallis(pd.DataFrame(global_results)); print(f"\n[INFO] Experimento Finalizado. Analíticas en: {self.base_dir}")

if __name__ == "__main__":
    app = EXPERIMENTACION_ANOVA(ACTIVE_CONFIG)
    app.run()


In [ ]:
!pip install -q transformers datasets peft scikit-learn torch accelerate spacy scipy xlsxwriter seaborn
!python -m spacy download en_core_web_sm -q
import os,sys,math,gc,json,warnings,shutil,types;from functools import partial;from itertools import product
import torch,torch.nn as nn,torch.nn.functional as F;from torch.utils.data import DataLoader
import numpy as np,pandas as pd,matplotlib.pyplot as plt,matplotlib as mpl;from mpl_toolkits.mplot3d.art3d import Line3DCollection;import seaborn as sns
from scipy.linalg import svd as scipy_svd;from scipy.stats import kruskal,kurtosis,gaussian_kde;from scipy.special import logsumexp;from tqdm.auto import tqdm;import spacy
import transformers;from transformers import AutoModelForCausalLM,AutoTokenizer,Trainer,TrainingArguments,DataCollatorForLanguageModeling
from peft import LoraConfig,get_peft_model;from datasets import load_dataset,Dataset
warnings.filterwarnings('ignore')
plt.rcParams.update({"font.family":"serif","font.serif":["cmr10","DejaVu Serif","Bitstream Vera Serif"],"mathtext.fontset":"cm","axes.labelsize":14,"font.size":12,"legend.fontsize":12,"xtick.labelsize":12,"ytick.labelsize":12,"figure.figsize":[10,8],"image.cmap":"jet","axes.formatter.use_mathtext":True})
try:from google.colab import drive;drive.mount('/content/drive');IN_COLAB=True
except ImportError:IN_COLAB=False
OUTS='L_kw_nL_Advanced';BASE_DRIVE_PATH=f'/content/drive/My Drive/University_MATH/{OUTS}' if IN_COLAB else f'./University_MATH/{OUTS}';DATASET_FULL_PATH=f'/content/drive/My Drive/University_MATH/x.json' if IN_COLAB else 'dummy.json'
QUICK_CONFIG={"run_config":{"experiment_name":"Exp_L_QQQ_Advanced","num_replicas":2,"device":"cuda" if torch.cuda.is_available() else "cpu","metric_for_hypothesis_testing":"RLCT"},"data_config":{"dataset_path":DATASET_FULL_PATH,"samples_in_balance":10,"finetune_budget":10,"max_seq_len":64},"model_config":{"base_model_id":"openai-community/gpt2","lora_config":{"r":4,"lora_alpha":8,"target_modules":["c_attn"],"fan_in_fan_out":True}},"training_config":{"max_steps":5,"learning_rate":1e-4,"per_device_train_batch_size":2,"logging_steps":1},"rlct_config":{"max_rlct_estimation_steps":5,"sgld_lr":1e-4},"lot_config":{"num_trajectories":50,"n_tokens":25,"use_alternating_source":True}}
INTERMEDIATE_CONFIG={"run_config":{"experiment_name":"Exp_PhD_2_Intermediate","num_replicas":2,"device":"cuda" if torch.cuda.is_available() else "cpu","metric_for_hypothesis_testing":"RLCT"},"data_config":{"dataset_path":DATASET_FULL_PATH,"samples_in_balance":100,"finetune_budget":50,"max_seq_len":128},"model_config":{"base_model_id":"openai-community/gpt2","lora_config":{"r":8,"lora_alpha":16,"target_modules":["c_attn"],"fan_in_fan_out":True}},"training_config":{"max_steps":20,"learning_rate":5e-5,"per_device_train_batch_size":4,"logging_steps":2},"rlct_config":{"max_rlct_estimation_steps":15,"sgld_lr":1e-4},"lot_config":{"num_trajectories":25,"n_tokens":10,"use_alternating_source":True}}
DEEP_CONFIG={"run_config":{"experiment_name":"Exp_PhD_3_Deep","num_replicas":3,"device":"cuda" if torch.cuda.is_available() else "cpu","metric_for_hypothesis_testing":"RLCT"},"data_config":{"dataset_path":DATASET_FULL_PATH,"samples_in_balance":500,"finetune_budget":150,"max_seq_len":128},"model_config":{"base_model_id":"openai-community/gpt2","lora_config":{"r":16,"lora_alpha":32,"target_modules":["c_attn"],"fan_in_fan_out":True}},"training_config":{"max_steps":50,"learning_rate":5e-5,"per_device_train_batch_size":8,"logging_steps":5},"rlct_config":{"max_rlct_estimation_steps":50,"sgld_lr":5e-5},"lot_config":{"num_trajectories":50,"n_tokens":15,"use_alternating_source":True}}
ACTIVE_CONFIG=QUICK_CONFIG
class CLASE_LOT_and_STATS:
    def __init__(self,master_config:dict):self.config=master_config;self.device=torch.device(self.config['run_config']['device']);self.base_dir=BASE_DRIVE_PATH;os.makedirs(self.base_dir,exist_ok=True);print(f"=== INICIALIZANDO EXPERIMENTO: {self.config['run_config']['experiment_name']} ===");print(f"Directorio de salida: {self.base_dir}");exec("try: self.nlp = spacy.load('en_core_web_sm')\nexcept OSError: import subprocess; subprocess.call([sys.executable,'-m','spacy','download','en_core_web_sm','-q']); self.nlp = spacy.load('en_core_web_sm')");self.tokenizer=AutoTokenizer.from_pretrained(self.config['model_config']['base_model_id']);self.tokenizer.pad_token=self.tokenizer.pad_token if self.tokenizer.pad_token is not None else self.tokenizer.eos_token;self.doc_gen=self.ExperimentDocumentationGenerator(self);self.data_handler=self.LinguisticDataHandler(self);self.lora_xs=self.LoRAXS_Arquitectura(self);self.rlct_estimator=self.SingularLearningRLCT(self);self.math_dynamics=self.MathematicalDynamics(self);self.visualizer=self.TrajectoryVisualizer(self);self.stats=self.StatisticalAnalysis(self);self.orchestrator=self.ExperimentOrchestrator(self);self.doc_gen.write_math_foundations()
    class ExperimentDocumentationGenerator:
        def __init__(self,parent):self.parent=parent;self.txt_dir=os.path.join(parent.base_dir,"Theoretical_Foundations");os.makedirs(self.txt_dir,exist_ok=True)
        def write_math_foundations(self):
          content=r"""\section{Fundamentos Matemáticos} ... """;
          with open(os.path.join(self.txt_dir,"Math_Foundations.txt"),"w",encoding="utf-8") as f:
              f.write(content.strip())
    class LinguisticDataHandler:
        def __init__(self,parent):self.parent=parent
        def prepare_dataset(self):
            cfg=self.parent.config['data_config']
            try:
                if os.path.exists(cfg['dataset_path']):
                  with open(cfg['dataset_path'],'r') as f:
                    data=json.load(f);df=pd.DataFrame(data)
                else:
                  raise FileNotFoundError
            except Exception as e:df=pd.DataFrame({"instruction":["Explain "+"word "*np.random.randint(5,20) for _ in range(cfg['samples_in_balance']*6)],"response":["This is a response "*5 for _ in range(cfg['samples_in_balance']*6)]})
            records=[];exec(compile('for idx,row in df.iterrows():\n doc=self.parent.nlp(str(row["instruction"]))\n words=[t.text.lower() for t in doc if t.is_alpha]\n records.append({**row,"ttr":len(set(words))/len(words) if words else 0.0})','<string>','exec'));df_ttr=pd.DataFrame(records).dropna(subset=['ttr']).sort_values('ttr').reset_index(drop=True);n_samples=cfg['samples_in_balance'];pools={"Low_Rich_Lexicon":Dataset.from_pandas(df_ttr.head(n_samples)),"Middle_Rich_Lexicon":Dataset.from_pandas(df_ttr.iloc[(len(df_ttr)-n_samples)//2:(len(df_ttr)-n_samples)//2+n_samples]),"High_Rich_Lexicon":Dataset.from_pandas(df_ttr.tail(n_samples))};return pools,Dataset.from_pandas(df_ttr)
    class LoRAXS_Arquitectura:
        def __init__(self,parent):self.parent=parent
        class SVD_Engine:
            @staticmethod
            def compute_svd(W:torch.Tensor,rank:int):U,S,Vt=scipy_svd(W.cpu().detach().float().numpy(),full_matrices=False);return torch.tensor(U[:,:rank]@np.diag(S[:rank]),dtype=W.dtype,device=W.device),torch.tensor(Vt[:rank,:],dtype=W.dtype,device=W.device)
        class LatentPatcher:
            @staticmethod
            def forward_latent(self_peft,x:torch.Tensor,*args,**kwargs):prev_dtype=x.dtype;result=self_peft.base_layer(x,*args,**kwargs);active_adapters=[getattr(self_peft,"active_adapter","default")] if isinstance(getattr(self_peft,"active_adapter","default"),str) else getattr(self_peft,"active_adapters",["default"]);exec(compile('for adapter in active_adapters:\n if adapter in self_peft.lora_A.keys() and self_peft.r.get(adapter,self_peft.r)>0:\n  x_type=x.to(self_peft.lora_A[adapter].weight.dtype)\n  proj_out=self_peft.lora_B[adapter](self_peft.default_lora_latent_mapping(self_peft.lora_A[adapter](self_peft.lora_dropout[adapter](x_type))))\n  result+=proj_out*(self_peft.scaling[adapter] if isinstance(self_peft.scaling,dict) else self_peft.scaling)','<string>','exec'));return result.to(prev_dtype)
        def transmute_to_loraxs(self,peft_model,rank):
            import peft;exec(compile('for name,target_module in peft_model.named_modules():\n if isinstance(target_module,peft.tuners.lora.Linear):\n  base_w=target_module.base_layer.weight.data.clone()\n  if getattr(target_module,"fan_in_fan_out",False):base_w=base_w.T\n  A_tensor,B_tensor=self.SVD_Engine.compute_svd(base_w,rank)\n  adapter=list(target_module.lora_A.keys())[0] if target_module.lora_A else "default"\n  dev,dtype=target_module.lora_A[adapter].weight.device,target_module.lora_A[adapter].weight.dtype\n  target_module.lora_A[adapter].weight=nn.Parameter(B_tensor.to(dev).to(dtype).contiguous())\n  target_module.lora_B[adapter].weight=nn.Parameter(A_tensor.to(dev).to(dtype).contiguous())\n  target_module.lora_A[adapter].weight.requires_grad=target_module.lora_B[adapter].weight.requires_grad=False\n  target_module.default_lora_latent_mapping=nn.Linear(rank,rank,bias=False).to(dev).to(dtype)\n  nn.init.normal_(target_module.default_lora_latent_mapping.weight,mean=0,std=1e-5)\n  target_module.forward=types.MethodType(self.LatentPatcher.forward_latent,target_module)','<string>','exec'));return peft_model
    class SingularLearningRLCT:
        def __init__(self,parent):self.parent=parent
        class SGLD(torch.optim.Optimizer):
            def __init__(self,params,lr=1e-4,noise_level=1.,temperature=1.):super().__init__(params,dict(lr=lr,noise_level=noise_level,temperature=temperature))
            def step(self):exec(compile('for group in self.param_groups:\n for p in group["params"]:\n  if p.grad is not None:\n   with torch.no_grad():\n    p.add_(p.grad,alpha=-0.5*group["lr"]/group["temperature"])\n    p.add_(torch.normal(mean=0.,std=group["noise_level"],size=p.size(),device=p.device),alpha=math.sqrt(group["lr"]))','<string>','exec'))
        def estimate_rlct(self,model,dataset):
            print("   -> RLCT Phase: Watanabe Estimator (SGLD)...");
            cfg=self.parent.config['rlct_config'];n=max(1,len(dataset));
            beta1,beta2=1.0/np.log(n),1.3/np.log(n);
            optimizer=self.SGLD(model.parameters(),lr=cfg['sgld_lr'],temperature=1.0/beta1);energies=[];model.train();
            exec(compile('for i in range(min(n,cfg["max_rlct_estimation_steps"])):\n sample=dataset[i%n];text=f"{sample["instruction"]}\\n{sample.get("response","")}"\n inputs=self.parent.tokenizer(text,return_tensors="pt",truncation=True,max_length=64).to(self.parent.device)\n optimizer.zero_grad();loss=model(**inputs,labels=inputs["input_ids"]).loss;loss.backward();optimizer.step()\n energies.append(loss.item()*n)','<string>','exec'));
            if not energies:
              return 0.0
            energies;V=np.array(energies);log_w=-(beta2-beta1)*V;norm_w=np.exp(log_w-logsumexp(log_w));rlct_lambda=(np.mean(V)-np.sum(norm_w*V))/(1/beta1-1/beta2)
            return 0.0 if np.isnan(rlct_lambda) or np.isinf(rlct_lambda) else rlct_lambda,energies
    class MathematicalDynamics:
        def __init__(self,parent):self.parent=parent
        def extract_and_compute(self,model,dataset):
            model.eval();traj=[];exec(compile('with torch.no_grad():\n for d in dataset.select(range(min(5,len(dataset)))):\n  tokens=self.parent.tokenizer(f"{d["instruction"]} {d.get("response","")}",return_tensors="pt",truncation=True,max_length=64).to(self.parent.device)\n  traj.append(np.stack([h[0,-1,:].cpu().numpy() for h in model(**tokens,output_hidden_states=True).hidden_states],axis=0))','<string>','exec'));
            if not traj:
              return None;
            M=np.stack(traj,axis=2);NL,_,_=M.shape;results={"Sigma":[],"Drift":[],"Diffusion":[]};U_prev=None;exec(compile('for t in range(NL):\n U,S,Vt=scipy_svd(M[t,:,:],full_matrices=False)\n results["Sigma"].append(S)\n if U_prev is not None:\n  results["Drift"].append(np.linalg.norm(U-U_prev,ord="fro"))\n  results["Diffusion"].append(np.var(M[t,:,:]-(U@np.diag(S)@Vt)))\n else:results["Drift"].append(0);results["Diffusion"].append(0)\n U_prev=U','<string>','exec'));
            return results
    class TrajectoryVisualizer:
        def __init__(self,parent):self.parent=parent;self.current_model=None;self.current_tokenizer=None;self.output_dir=None
        def plot_rlct(self,energies,method):plt.figure(figsize=(6,4));plt.plot(energies,marker='o',linestyle='-',color='indigo');plt.title(f"Estimador Watanabe SGLD - {method}");plt.xlabel("Pasos SGLD");plt.ylabel(r"Energía $nL_n(w)$");plt.grid(True,alpha=0.3);plt.savefig(os.path.join(self.output_dir,"rlct_evolution_lambda.png"),bbox_inches='tight');plt.close()
        def plot_math_dynamics(self,math_res):
            if not math_res:return
            fig,ax=plt.subplots(1,3,figsize=(15,4));exec(compile('for i,S in enumerate(math_res["Sigma"]):ax[0].plot(S[:10],label=f"Layer {i}" if i%2==0 else "")','<string>','exec'));ax[0].set_title(r"Decaimiento $\Sigma(t)$");ax[0].set_yscale('log');ax[1].plot(math_res["Drift"],marker='s',color='r');ax[1].set_title(r"Deriva $\|A(t)\|_F$");ax[2].plot(math_res["Diffusion"],marker='^',color='g');ax[2].set_title(r"Difusión");ax[2].set_yscale('log');plt.tight_layout();plt.savefig(os.path.join(self.output_dir,"Mathematical_Dynamics.png"),bbox_inches='tight');plt.close()
        def generate_trajectories(self,text,lot_config):
            print(f"   -> Generating {lot_config['num_trajectories']} LoT trajectories...");tokens=self.current_tokenizer(text,return_tensors='pt',truncation=True,max_length=1024).input_ids[0];sentences=[tokens[i:i+lot_config['n_tokens']]
            for i in range(0,len(tokens)-lot_config['n_tokens']+1,lot_config['n_tokens'])][:lot_config['num_trajectories']];
            if not sentences:
              return None
            trajectories=[];self.current_model.eval();exec(compile('with torch.no_grad():\n for s in tqdm(sentences,desc="Tracing",leave=False):\n  out=self.current_model(s.unsqueeze(0).to(self.current_model.device),output_hidden_states=True)\n  trajectories.append(np.stack([h[0,-1,:].float().cpu().numpy() for h in out.hidden_states],axis=1))','<string>','exec'));return np.stack(trajectories,axis=2)[:,1:,:] if trajectories else None
        def _plot_ribbon_trajectories(self, traj):
            h, l, s = traj.shape; flat = traj.reshape(h, -1).astype(np.float32)
            try: U, S, V = scipy_svd(flat, full_matrices=False)
            except: return
            proj = (np.diag(S[:3]) @ V[:3, :]).reshape(3, l, s); fig = plt.figure(figsize=(10, 8)); ax = fig.add_subplot(111, projection='3d'); cmap = plt.get_cmap('jet', l)
            for i in range(min(s, 50)):
                points = np.array([proj[0,:,i], proj[1,:,i], proj[2,:,i]]).T.reshape(-1, 1, 3)
                ax.add_collection(Line3DCollection(np.concatenate([points[:-1], points[1:]], axis=1), cmap=cmap, norm=plt.Normalize(0, l), alpha=0.4, linewidth=0.8))
            ax.set_xlim(proj[0].min(), proj[0].max()); ax.set_ylim(proj[1].min(), proj[1].max()); ax.set_zlim(proj[2].min(), proj[2].max())
            ax.set_xlabel(r'$\mathbf{u}_1$', labelpad=10); ax.set_ylabel(r'$\mathbf{u}_2$', labelpad=10); ax.set_zlabel(r'$\mathbf{u}_3$', labelpad=10)
            ax.set_title(r'Lines of Thought Manifold (PCA)', y=1.02); ax.grid(False)
            fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, l)), ax=ax, shrink=0.6, pad=0.1).set_label(r'Layer Depth $t$', rotation=270, labelpad=20)
            plt.savefig(f"{self.output_dir}/lot_manifold_pca_projection.png", dpi=300, bbox_inches='tight'); plt.close()
        #def _plot_ribbon_trajectories(self,traj):h,l,s=traj.shape;flat=traj.reshape(h,-1).astype(np.float32);exec(compile('try:U,S,V=scipy_svd(flat,full_matrices=False)\nexcept:return','<string>','exec'));proj=(np.diag(S[:3])@V[:3,:]).reshape(3,l,s);fig=plt.figure(figsize=(10,8));ax=fig.add_subplot(111,projection='3d');cmap=plt.get_cmap('jet',l);exec(compile('for i in range(min(s,50)):\n points=np.array([proj[0,:,i],proj[1,:,i],proj[2,:,i]]).T.reshape(-1,1,3)\n ax.add_collection(Line3DCollection(np.concatenate([points[:-1],points[1:]],axis=1),cmap=cmap,norm=plt.Normalize(0,l),alpha=0.4,linewidth=0.8))','<string>','exec'));ax.set_xlim(proj[0].min(),proj[0].max());ax.set_ylim(proj[1].min(),proj[1].max());ax.set_zlim(proj[2].min(),proj[2].max());ax.set_xlabel(r'$\mathbf{u}_1$',labelpad=10);ax.set_ylabel(r'$\mathbf{u}_2$',labelpad=10);ax.set_zlabel(r'$\mathbf{u}_3$',labelpad=10);ax.set_title(r'Lines of Thought Manifold (PCA)',y=1.02);ax.grid(False);fig.colorbar(plt.cm.ScalarMappable(cmap=cmap,norm=plt.Normalize(0,l)),ax=ax,shrink=0.6,pad=0.1).set_label(r'Layer Depth $t$',rotation=270,labelpad=20);plt.savefig(f"{self.output_dir}/lot_manifold_pca_projection.png",dpi=300,bbox_inches='tight');plt.close()
        def _plot_coordinate_manifold(self,traj):h,l,s=traj.shape;vars_dim=np.var(traj.reshape(h,-1),axis=1);top=np.argsort(vars_dim)[-3:];fig=plt.figure(figsize=(10,8));ax=fig.add_subplot(111,projection='3d');cmap=plt.get_cmap('viridis',l);exec(compile('for t in range(l):ax.scatter(traj[top[0],t,::2],traj[top[1],t,::2],traj[top[2],t,::2],color=cmap(t),s=3,alpha=0.6)','<string>','exec'));ax.set_xlabel(rf'$\mathbf{{e}}_{{{top[0]}}}$');ax.set_ylabel(rf'$\mathbf{{e}}_{{{top[1]}}}$');ax.set_zlabel(rf'$\mathbf{{e}}_{{{top[2]}}}$');ax.set_title(r'Manifold in Coordinate Basis',y=1.02);fig.colorbar(plt.cm.ScalarMappable(cmap=cmap,norm=mpl.colors.Normalize(vmin=0,vmax=l)),ax=ax,shrink=0.6,pad=0.1).set_label(r'Layer Depth $t$',rotation=270,labelpad=20);plt.savefig(f"{self.output_dir}/lot_coordinate_basis_projection.png",dpi=300,bbox_inches='tight');plt.close()
        def _plot_sv_angles(self,traj):
          n=traj.shape[1];us=[scipy_svd(traj[:,t,:].astype(np.float32),full_matrices=False)[0] for t in range(n)];max_i=min(4,us[0].shape[1]);
          if max_i<1:
            return;fig,axes=plt.subplots(2,2,figsize=(12,10));fig.suptitle(r'Angle $\arccos(\mathbf{u}_i^{(t_1)} \cdot \mathbf{u}_i^{(t_2)})$',y=0.95,fontsize=16);exec(compile('for i in range(max_i):\n mat=np.array([[np.degrees(np.arccos(np.clip(np.abs(np.dot(us[t1][:,i],us[t2][:,i])),-1.0,1.0))) for t2 in range(n)] for t1 in range(n)])\n im=axes[i//2,i%2].imshow(mat,cmap="viridis",origin="lower",vmin=0,vmax=90)\n axes[i//2,i%2].set_title(rf"Singular Vector $u_{{{i+1}}}$")\n fig.colorbar(im,ax=axes[i//2,i%2],fraction=0.046,pad=0.04).set_label(r"$\\theta$ (deg)",rotation=270,labelpad=15)','<string>','exec'));plt.tight_layout(rect=[0,0.03,1,0.93]);plt.savefig(f"{self.output_dir}/svd_singular_vector_alignment_angles.png",dpi=300);plt.close()
        def _plot_singular_values(self,traj):n=traj.shape[1];sigs=[scipy_svd(traj[:,t,:].astype(np.float32),full_matrices=False,compute_uv=False) for t in range(n)];fig,ax=plt.subplots(figsize=(10,8));cmap=plt.get_cmap('jet',n);exec(compile('for t in range(n):ax.loglog(range(1,len(sigs[t])+1),sigs[t],color=cmap(t),alpha=0.9,linewidth=1.5)','<string>','exec'));ax.set_title(r'Singular Value Spectrum');ax.set_xlabel(r'Rank $k$');ax.set_ylabel(r'$\sigma_k$');ax.grid(True,which="both",ls="-",alpha=0.2);fig.colorbar(plt.cm.ScalarMappable(cmap=cmap,norm=mpl.colors.Normalize(vmin=0,vmax=n)),ax=ax).set_label(r'Layer $t$',rotation=270,labelpad=20);plt.savefig(f"{self.output_dir}/svd_singular_value_spectrum_decay.png",dpi=300,bbox_inches='tight');plt.close()
        def _plot_kl_vs_k(self,traj):
            embeds=self.current_model.get_output_embeddings();exec(compile('if embeds is None and hasattr(self.current_model,"base_model"):embeds=self.current_model.base_model.get_output_embeddings()','<string>','exec'));
            if embeds is None:
              return
            final=torch.tensor(traj[:,-1,:].T).to(self.current_model.device).float();head=embeds.weight.T.float();kl=[];exec(compile('with torch.no_grad():\n true_p=torch.softmax(final@head,dim=-1);U=torch.svd(final.T)[0]\n max_rank=min(U.shape[1],final.shape[1],128)\n if max_rank<2:return\n K_dims=np.unique(np.geomspace(1,max(2,max_rank-1),10,dtype=int))\n for k in K_dims:\n  kl.append(torch.nn.functional.kl_div(torch.log_softmax(((U[:,:k]@U[:,:k].T)@final.T).T@head,dim=-1),true_p,reduction="batchmean").item())','<string>','exec'));fig,ax=plt.subplots(figsize=(8,6));ax.plot(K_dims,kl,'k.-',linewidth=1.5);ax.set_title(r'$\langle D_{KL}(\mathbf{p}_K^\nu || \mathbf{p}^\nu) \rangle$ vs Rank $K$');plt.savefig(f"{self.output_dir}/kl_divergence_information_loss_vs_rank.png",dpi=300,bbox_inches='tight');plt.close()
        def _plot_delta_heatmaps(self,traj):n=traj.shape[1];mat_m,mat_v,mat_k=np.full((n,n),np.nan),np.full((n,n),np.nan),np.full((n,n),np.nan);exec(compile('for t in range(n):\n for tau in range(1,n-t):\n  d=traj[:,t+tau,:]-traj[:,t,:]\n  mat_m[t+tau,t]=np.mean(d)/(np.std(d)+1e-9)\n  mat_v[t+tau,t]=np.log(np.var(d)+1e-9)\n  mat_k[t+tau,t]=np.mean(kurtosis(d,axis=1))','<string>','exec'));fig,axes=plt.subplots(1,3,figsize=(18,5));labs=[r'$\langle |\mu/\sigma| \rangle$',r'$\langle \log(\sigma^2) \rangle$',r'$\langle |\kappa| \rangle$'];mats=[mat_m,mat_v,mat_k];exec(compile('for i in range(3):\n im=axes[i].imshow(mats[i],cmap="jet",aspect="auto",origin="upper")\n axes[i].set_title(labs[i])\n fig.colorbar(im,ax=axes[i])','<string>','exec'));plt.savefig(f"{self.output_dir}/drift_diffusion_statistics_heatmaps.png",dpi=300,bbox_inches='tight');plt.close()
        def _plot_kde_residuals(self,traj):
            h,l,s=traj.shape;fig,ax=plt.subplots(figsize=(8,6));idx8=min(8,l-1);idx10=min(10,l-1);d8,d10=traj[:,idx8,:]-traj[:,0,:],traj[:,idx10,:]-traj[:,0,:];sns.kdeplot((d8.flatten()-np.mean(d8))/(np.std(d8)+1e-9),ax=ax,label=rf'$t+\tau={idx8}$',color='tab:blue');sns.kdeplot((d10.flatten()-np.mean(d10))/(np.std(d10)+1e-9),ax=ax,label=rf'$t+\tau={idx10}$',color='tab:orange');x=np.linspace(-5,5,100);ax.plot(x,1/np.sqrt(2*np.pi)*np.exp(-0.5*x**2),'k--',label='Gaussian');ax.set_yscale('log');ax.legend();plt.savefig(f"{self.output_dir}/residual_distribution_normality_check.png",dpi=300,bbox_inches='tight');plt.close()
            exec(compile('try:\n flat=traj.reshape(h,-1).astype(np.float32);U,S,V=scipy_svd(flat,full_matrices=False)\n proj=(np.diag(S[:2])@V[:2,:]).reshape(2,l,s);fig,axes=plt.subplots(2,5,figsize=(20,8));shifts=[6,7,8,9,10]\n for i,sh in enumerate(shifts):\n  if sh<l:\n   for row,t_idx in enumerate([sh,max(0,sh-2)]):\n    u1,u2=proj[0,t_idx,:],proj[1,t_idx,:]\n    if u1.min()==u1.max() or u2.min()==u2.max():continue\n    k=gaussian_kde([u1,u2]);xi,yi=np.mgrid[u1.min():u1.max():20j,u2.min():u2.max():20j]\n    axes[row,i].contour(xi,yi,k(np.vstack([xi.flatten(),yi.flatten()])).reshape(xi.shape),cmap="jet")\n plt.savefig(f"{self.output_dir}/probability_density_evolution_contours.png",dpi=300,bbox_inches="tight");plt.close()\nexcept Exception as e:print(f"      [Warning] Contour KDE failed: {e}")','<string>','exec'))
        def run_full_analysis(self,lot_config,dolly_dataset):
            print(f"   -> Executing LoT Analysis in: {os.path.basename(self.output_dir)}");sample=dolly_dataset.shuffle().select(range(min(len(dolly_dataset),100))) if lot_config.get('use_alternating_source',True) else None;text="\n\n".join(list(sample['instruction'])+[r for r in sample['response'] if r]) if sample else " ".join(["thought"]*500);traj=self.generate_trajectories(text,lot_config);exec(compile('if traj is not None:\n plot_methods=[m for m in dir(self) if m.startswith("_plot_") and callable(getattr(self,m))]\n for m in plot_methods:\n  try:getattr(self,m)(traj)\n  except Exception as e:print(f"      [Warning] Plot method {m} failed: {e}")\n print("   -> Scientific Charts Generated.")','<string>','exec'))
    class StatisticalAnalysis:
        def __init__(self,parent):self.parent=parent
        def run_kruskal_wallis(self,df_results):pass
    class ExperimentOrchestrator:
        def __init__(self,parent):self.parent=parent
        @staticmethod
        def tokenize_batch_fn(batch,tokenizer,max_len):resps=batch.get('response',['']*len(batch['instruction']));texts=[f"{inst}\\n{resp}" for inst,resp in zip(batch['instruction'],resps)];return tokenizer(texts,truncation=True,padding="max_length",max_length=max_len)
        @staticmethod
        def plot_training_history(history,run_id,output_dir):
            if history:
                df=pd.DataFrame(history).dropna(subset=['loss'])
                if not df.empty:plt.figure(figsize=(10,6));plt.plot(df['step'],df['loss'],marker='o',label='Training Loss');plt.title(rf"Training Loss Run {run_id}");plt.yscale('log');plt.grid(True,alpha=0.3);plt.savefig(os.path.join(output_dir,"training_history_loss.png"),bbox_inches='tight');plt.close()
        def calculate_ppl(self,model,dataset):
            model.eval();total_loss=0;count=0;exec(compile('with torch.no_grad():\n for i in range(min(5,len(dataset))):\n  inputs=self.parent.tokenizer(dataset[i]["instruction"],return_tensors="pt",truncation=True,max_length=64).to(self.parent.device)\n  loss=model(**inputs,labels=inputs["input_ids"]).loss\n  if not torch.isnan(loss):total_loss+=loss.item();count+=1','<string>','exec'));avg_loss=total_loss/count if count>0 else 100.0;return math.exp(min(avg_loss,20))
        def run(self):
            cfg=self.parent.config;pools,_=self.parent.data_handler.prepare_dataset();global_results=[];max_len=cfg['data_config']['max_seq_len'];tokenize_fn=partial(self.tokenize_batch_fn,tokenizer=self.parent.tokenizer,max_len=max_len);exec(compile('for rep in range(cfg["run_config"]["num_replicas"]):\n for treat_name,pool_ds in pools.items():\n  tokenized_pool_ds=pool_ds.map(tokenize_fn,batched=True)\n  for method in ["LORA_Standard","LORA_XS"]:\n   run_id=f"Rep{rep}_{treat_name}_{method}"\n   print(f"\\n[{run_id}] Iniciando Pipeline...")\n   run_dir=os.path.join(self.parent.base_dir,run_id);os.makedirs(run_dir,exist_ok=True)\n   base_model=AutoModelForCausalLM.from_pretrained(cfg["model_config"]["base_model_id"]).to(self.parent.device)\n   peft_cfg=LoraConfig(task_type="CAUSAL_LM",**cfg["model_config"]["lora_config"])\n   peft_model=get_peft_model(base_model,peft_cfg)\n   if method=="LORA_XS":peft_model=self.parent.lora_xs.transmute_to_loraxs(peft_model,rank=peft_cfg.r)\n   trainer=Trainer(model=peft_model,args=TrainingArguments(output_dir=os.path.join(run_dir,"ckpt"),max_steps=cfg["training_config"]["max_steps"],learning_rate=cfg["training_config"]["learning_rate"],per_device_train_batch_size=cfg["training_config"]["per_device_train_batch_size"],logging_steps=cfg["training_config"]["logging_steps"],report_to="none",save_strategy="no",remove_unused_columns=True),train_dataset=tokenized_pool_ds,data_collator=DataCollatorForLanguageModeling(self.parent.tokenizer,mlm=False))\n   trainer.train()\n   self.plot_training_history(trainer.state.log_history,run_id,run_dir)\n   ppl=self.calculate_ppl(peft_model,pool_ds)\n   self.parent.visualizer.current_model=peft_model;self.parent.visualizer.current_tokenizer=self.parent.tokenizer;self.parent.visualizer.output_dir=run_dir\n   rlct_lambda,energies=self.parent.rlct_estimator.estimate_rlct(peft_model,pool_ds)\n   self.parent.visualizer.plot_rlct(energies,method)\n   math_res=self.parent.math_dynamics.extract_and_compute(peft_model,pool_ds)\n   self.parent.visualizer.plot_math_dynamics(math_res)\n   self.parent.visualizer.run_full_analysis(cfg["lot_config"],pool_ds)\n   global_results.append({"Replica":rep,"Treatment":treat_name,"Method":method,"RLCT":rlct_lambda,"PPL":ppl})\n   del peft_model,base_model,trainer;gc.collect();torch.cuda.empty_cache()','<string>','exec'));df_res=pd.DataFrame(global_results);df_res.to_excel(os.path.join(self.parent.base_dir,"LOT_and_STATS_Results.xlsx"),index=False);exec(compile('try:self.parent.stats.run_kruskal_wallis(df_res)\nexcept:pass','<string>','exec'));print(f"\\n[INFO] Experimento Completado. Archivos generados en: {self.parent.base_dir}")
if __name__=="__main__":app=CLASE_LOT_and_STATS(ACTIVE_CONFIG);app.orchestrator.run()